In [1]:
# stop

# FINAL FAST SYSTEM WITH BLACKLIST AND MANUAL CONTROL
# Import necessary libraries
import serial
import time
import requests
import cv2
import os
import numpy as np
import urllib.request
import sys

# Motor control configurations
arduino_port = '/dev/cu.usbmodem1402'  # Update this to match your port
baud_rate = 9600  # Match this with the Arduino code
use_wifi = True  # Set to True to use WiFi, False to use Serial
arduino_ip = "http://172.20.10.5:80"  # Replace with Arduino's IP address

# Dataset and face recognizer configurations
dataset_path = "/Users/samuelyeo/Desktop/enrolled_faces"
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("trainer.yml")
f_cas = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Google Apps Script URLs
# Image
#google_script_url = 'https://script.google.com/macros/s/AKfycbwamcaNQzChDe6aP74EJOlGtnSqPKZzMQSK4-vzeKBMZIrOV63KaC7I2-Ct41MvxaE9mg/exec'
# Integrated version:
google_script_url = 'https://script.google.com/macros/s/AKfycbx6IllJk4_d_OVm8ezte71PbELEZIgeB6kSZCDD9Or-W1DlUHTLlscMjZPDITI27sXQ2Q/exec'


# RFID
#WEB_APP_URL = "https://script.google.com/macros/s/AKfycbzgKIZ9V-KPHkK19migHKxcqa9n5_mnBaO1J1RZbC2Mp2BD-XwgjAHwly_-cJd6h10RSg/exec"
# google_script_url for images and WEB_APP_URL for passport
# Integrated version
WEB_APP_URL = "https://script.google.com/macros/s/AKfycbx0SFiPmZ6HZs28xy-cAgYlr8NpJkEctaAu5YMeY_1BWf_1kDYoYwPJhp6Yt4dtfUv1JA/exec"


# Function to control stepper motor via Serial with try-except
def send_pass_serial():
    try:
        ser = serial.Serial(arduino_port, baud_rate)
        time.sleep(60)  # Initialize connection
        ser.write(b"PASS\n")
        print("Command sent via Serial.")
        time.sleep(60)  # Wait for motor movement to complete
        ser.close()
    except serial.SerialException as e:
        print(f"Serial connection error: {e}") 

# Function to control stepper motor via WiFi
def send_pass_wifi():
    try:
        response = requests.get(f"{arduino_ip}/PASS", timeout=5)
        if response.status_code == 200:
            print("Command sent via WiFi successfully.")
        else:
            print("Failed to send command via WiFi.")
        time.sleep(60)
    except requests.RequestException:
        print("\n")


# Load labels from the dataset path
def create_label_dict(dataset_path):
    label_dict = {}
    label_counter = 1
    for filename in os.listdir(dataset_path):
        if filename.endswith(".jpg"):
            name = os.path.splitext(filename)[0]
            label_dict[label_counter] = name
            label_counter += 1
    return label_dict

label_dict = create_label_dict(dataset_path)

# Check Google Apps Script for "pass" or "fail"
def check_for_match():
    try:
        response = requests.get(WEB_APP_URL)
        response_data = response.json()
        
        # Check if the response has "pass" or "fail"
        if response_data.get("result") == "pass":
            print("Pass detected from Google Apps Script.")
            return True
        else:
            print("Waiting for pass...")
            return False
            
    except Exception as e:
        print("Error:", e)
        return False

# Recognize face and trigger motor if "pass" is returned
def recognize_face(img, gray):
    faces = f_cas.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        label, confidence = recognizer.predict(roi_gray)
        
        if confidence > 100:
            label = "Unknown"
        else:
            label = label_dict.get(label, "Unknown")
        
        print(f"Recognized face: {label}")

        # Blacklist component
        if label.lower().startswith('b'):
            print("Blacklisted face detected. Terminating program.")
            cv2.destroyAllWindows()  # Close any open CV windows
            sys._exit(0)
        # End of blacklist component
        
        # Send face label to Google Sheets and check for "pass"
        params = {'faceLabel': label}
        response = requests.post(google_script_url, data=params)
        
        print(f"Google Sheets response: {response.text}")
        
        if response.text.strip().lower() == "pass":
            if use_wifi:
                send_pass_wifi()
            else:
                send_pass_serial()
            print("Pass detected. Executed motor command.")
            return True

    return False

# Main function for facial recognition and motor execution
# Main function for facial recognition and motor execution
def start_facial_recognition():
    url = 'http://172.20.10.6/320x240.jpg'
    while True:
        img_resp = urllib.request.urlopen(url)
        imgnp = np.array(bytearray(img_resp.read()), dtype=np.uint8)
        img = cv2.imdecode(imgnp, -1)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Recognize faces in the image
        if recognize_face(img, gray):
            print("Motor operation complete. Returning to 'waiting for pass...'")
            return  # Exit to allow the main loop to start waiting for "pass" again
        
        # Display live transmission
        cv2.imshow("Live Transmission", img)
        
        # Check for key presses
        key = cv2.waitKey(5)
        if key == ord('q'):  # Press 'q' to manually quit
            cv2.destroyAllWindows()
            exit()
        elif key == ord('l'):  # Press 'l' to trigger the motor immediately
            print("Manual trigger activated (key 'l').")
            if use_wifi:
                send_pass_wifi()
            else:
                send_pass_serial()
            print("Motor function executed manually.")
            return  # Return to the main loop after manual trigger
            cv2.destroyAllWindows()
            os.exit(1)


# Main program loop
while True:
    # Run check_for_match in a loop until a "pass" is detected
    while not check_for_match():
        time.sleep(1)  # Run every 1 second
    
    print("Starting facial recognition...")
    start_facial_recognition()


Pass detected from Google Apps Script.
Starting facial recognition...


2024-11-27 15:59:06.350 python[89307:5044947] +[IMKClient subclass]: chose IMKClient_Legacy
2024-11-27 15:59:06.350 python[89307:5044947] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Google Sheets response: fail
Recognized face: Unknown
Goo

URLError: <urlopen error [Errno 60] Operation timed out>